# Workshop: Build & Deploy a Multi-Agent System on Amazon Bedrock AgentCore

In this workshop you will **build a multi-agent market surveillance system from scratch**, then **deploy it to Amazon Bedrock AgentCore Runtime** - a fully managed container runtime that removes the need to provision or manage servers.

> 📖 **Companion blog post:** [Market Surveillance Agent with LangGraph and Strands on AgentCore](https://aws.amazon.com/blogs/machine-learning/market-surveillance-agent-with-langgraph-and-strands-on-agentcore/) <!-- Update with final URL -->
> This notebook is the hands-on companion to that post. The blog explains the *why* (architecture decisions, the LangGraph + Strands split, the secure tool design); this notebook walks you through the *how*, cell by cell.

The workshop is split into two parts:

| Part | What you will do |
|------|-----------------|
| **Part 1 - Build from scratch** | Create tools, wire up a Strands agent, add an orchestrator, and connect everything with a LangGraph state graph |
| **Part 2 - Scale up & deploy** | Import the full six-agent system and deploy it to AgentCore with one command |

### Architecture

```
                         +--------------------------------------+
                         |  Amazon Bedrock AgentCore Runtime    |
                         |                                      |
  User --> invoke -->    |  LangGraph Workflow                  |
                         |    |- Orchestrator  (routes queries) |
                         |    |- SecurityMonitor (intraday)     |
                         |    |- BrokerMonitor   (multi-day)    |
                         |    |- RiskMonitor     (broker focus) |
                         |    |- IntelAnalyst    (web research) |
                         |    +- Synthesizer     (final report) |
                         +--------------------------------------+
```

**Frameworks used:** [Strands Agents SDK](https://github.com/strands-agents/sdk-python) for individual agents, [LangGraph](https://langchain-ai.github.io/langgraph/) for orchestration, [Amazon Bedrock](https://aws.amazon.com/bedrock/) for model inference, and [AgentCore Runtime](https://docs.aws.amazon.com/bedrock/latest/userguide/agentcore.html) for managed deployment.

## Prerequisites

| Requirement | Notes |
|---|---|
| Python 3.11+ | Required by LangGraph and Strands Agents (3.10 works, but 3.11+ is recommended) |
| AWS credentials | `aws sts get-caller-identity` should succeed |
| Bedrock model access | Enable `anthropic.claude-sonnet-4-6` in your target region ([request access](https://console.aws.amazon.com/bedrock/home#/modelaccess)) |
| Docker | Not required — the build runs in the cloud via AWS CodeBuild |

**Required IAM permissions:**

| Service | Actions | Purpose |
|---------|---------|---------|
| `bedrock` | `InvokeModel`, `InvokeModelWithResponseStream`, `ListFoundationModels` | Model inference |
| `ecr` | `CreateRepository`, `GetAuthorizationToken`, `BatchCheckLayerAvailability`, `InitiateLayerUpload`, `UploadLayerPart`, `CompleteLayerUpload`, `PutImage` | Container image registry |
| `codebuild` | `CreateProject`, `StartBuild`, `BatchGetBuilds` | ARM64 image build |
| `iam` | `CreateRole`, `AttachRolePolicy`, `PutRolePolicy`, `PassRole` | Execution role for the runtime |
| `bedrock-agentcore` | `CreateMemory`, `GetMemory`, `ListMemories`, `CreateAgentRuntime`, `InvokeAgentRuntime`, `GetAgentRuntimeEndpoint`, `DeleteAgentRuntime` | Memory + managed runtime lifecycle |
| `s3` | `CreateBucket`, `PutObject`, `GetObject` | CodeBuild source bucket |

> **Cost notice:** Running this notebook creates billable AWS resources (CodeBuild, ECR, AgentCore Memory, an AgentCore Runtime endpoint, and Bedrock inference on every invocation). Run the **Cleanup** section at the end to stop ongoing charges.

---
## 1. Setup

In [ ]:
# Install dependencies - restart the kernel if imports fail after this cell
%pip install -r ../requirements.txt

In [ ]:
import boto3
import json
from pathlib import Path
from boto3.session import Session

# ---- Region & model configuration (edit these) -----------------------------
# Notebook uses Claude Sonnet 4.6; The model is reached through cross-region inference
# profiles whose geo prefix - us. / eu. / apac. - MUST match your region: a
# "us." profile only works from a us-* region. Set REGION to match, or change
# the model IDs to the prefix for your region.
REGION = "us-east-1"
MODEL_ID = "us.anthropic.claude-sonnet-4-6"


boto_session = Session()
region = REGION or boto_session.region_name or "us-east-1"

sts = boto3.client("sts", region_name=region)
identity = sts.get_caller_identity()


# ---- Guard: fail fast if a model isn't reachable from this region -----------
def ensure_model_available(model_id: str, region: str) -> None:
    """Raise a clear error if model_id can't be used from region (wrong geo
    prefix, or the inference profile isn't present in this region)."""
    geo = model_id.split(".")[0]  # 'us', 'eu', 'apac'
    if "." in model_id and not region.startswith(f"{geo}-"):
        raise ValueError(
            f"Model {model_id!r} is a '{geo}.' inference profile, but REGION is "
            f"{region!r}. Cross-region profiles only work from a matching region "
            f"('{geo}.' -> {geo}-* regions). Fix REGION or the model IDs at the top of this cell."
        )
    bedrock = boto3.client("bedrock", region_name=region)
    available, token = set(), None
    while True:
        kwargs = {"nextToken": token} if token else {}
        resp = bedrock.list_inference_profiles(**kwargs)
        available.update(p["inferenceProfileId"] for p in resp.get("inferenceProfileSummaries", []))
        token = resp.get("nextToken")
        if not token:
            break
    if model_id not in available:
        raise ValueError(
            f"Model {model_id!r} is not available in region {region!r}. Confirm the "
            f"region is correct and enable model access in the Bedrock console, then "
            f"re-run: https://console.aws.amazon.com/bedrock/home?region={region}#/modelaccess"
        )

ensure_model_available(MODEL_ID, region)

print(f"Region        : {region}")
print(f"Model  : {MODEL_ID}")


Region        : us-east-1
Part 1 model  : us.anthropic.claude-sonnet-4-6
Deploy model  : us.anthropic.claude-opus-4-8


---
# Part 1 - Build a Surveillance Agent from Scratch

In this part you will:
1. Define a mock market-data layer
2. Wrap it with Strands `@tool` functions
3. Create a single surveillance agent that uses the tools
4. Add an orchestrator agent for routing
5. Wire everything together with a LangGraph state graph

## 2. Define the Data Layer

Real surveillance systems query databases. For this workshop we use in-memory CSV strings that simulate two market-data tables. The agent tools will query this data as if it were a real database.

In [ ]:
# - Mock market data --------------------------------------------
# In production these would be database tables.  The agent doesn't
# know the data is fake - it discovers the schema via tools.

MOCK_DATA = {
    "TradeActivity": (
        "date,time,symbol,price,volume,side,broker\n"
        "2024-03-15,09:35:00,AAPL,172.50,1500,BUY,ALPHA_CAPITAL\n"
        "2024-03-15,09:35:02,AAPL,172.55,800,BUY,SUMMIT_TRADING\n"
        "2024-03-15,09:36:15,AAPL,172.30,2200,SELL,VERTEX_SECURITIES\n"
        "2024-03-15,10:00:05,AAPL,173.00,2500,BUY,SUMMIT_TRADING\n"
        "2024-03-15,10:15:00,AAPL,173.10,1100,SELL,CRESTVIEW_CAPITAL\n"
        "2024-03-15,10:30:00,AAPL,173.25,3200,BUY,PINNACLE_PARTNERS\n"
        "2024-03-15,09:40:10,MSFT,425.20,3000,BUY,PINNACLE_PARTNERS\n"
        "2024-03-15,10:05:30,MSFT,425.50,4000,BUY,VERTEX_SECURITIES\n"
        "2024-03-15,10:20:45,MSFT,425.80,2800,BUY,ALPHA_CAPITAL"
    ),
    "PriceBars": (
        "date,time,symbol,high,low,volume,num_trades,vwap\n"
        "2024-03-15,09:30:00,AAPL,172.80,171.90,15000,45,172.35\n"
        "2024-03-15,10:00:00,AAPL,173.20,172.10,22000,67,172.65\n"
        "2024-03-15,10:30:00,AAPL,173.50,172.80,18000,52,173.15\n"
        "2024-03-15,11:00:00,AAPL,173.80,173.20,14000,41,173.50\n"
        "2024-03-15,09:30:00,MSFT,425.50,424.80,12000,38,425.15\n"
        "2024-03-15,10:00:00,MSFT,426.00,425.20,19000,55,425.60"
    ),
}

# - Report catalog ----------------------------------------------
# Agents call get_report_list() to discover what data is available.

AVAILABLE_REPORTS = [
    {
        "report_name": "TradeActivity",
        "description": "Tick-level trade data with price, volume, side, and executing broker.",
    },
    {
        "report_name": "PriceBars",
        "description": "Aggregated price bars with high, low, volume, num_trades, and VWAP.",
    },
]

# - Report schemas ----------------------------------------------
# Agents call get_report_schema() to learn column names before
# building SQL queries.

REPORT_SCHEMAS = {
    "TradeActivity": {
        "displayName": "Trades",
        "reportName": "TradeActivity",
        "parameters": [
            {"name": "date", "description": "Trading date (YYYY-MM-DD)", "required": True},
            {"name": "symbol", "description": "Stock ticker", "required": False},
        ],
        "columns": [
            {"name": "date"},  {"name": "time"},   {"name": "symbol"},
            {"name": "price"}, {"name": "volume"}, {"name": "side"},
            {"name": "broker"},
        ],
    },
    "PriceBars": {
        "displayName": "Price Bars",
        "reportName": "PriceBars",
        "parameters": [
            {"name": "date", "description": "Trading date (YYYY-MM-DD)", "required": True},
            {"name": "symbol", "description": "Stock ticker", "required": True},
        ],
        "columns": [
            {"name": "date"},       {"name": "time"},   {"name": "symbol"},
            {"name": "high"},       {"name": "low"},    {"name": "volume"},
            {"name": "num_trades"}, {"name": "vwap"},
        ],
    },
}

print(f"Loaded {len(MOCK_DATA)} tables, {sum(d.count(chr(10)) for d in MOCK_DATA.values())} rows total")

Loaded 2 tables, 15 rows total


## 3. Create Agent Tools

Strands agents discover their capabilities through **tools** - Python functions decorated with `@tool`. Each tool's docstring is what the LLM reads to decide when and how to call it.

Our agent uses a three-step pipeline:

```
get_report_list  ->  get_report_schema  ->  run_report
  "what data          "what columns        "filter + fetch the rows
   exists?"            does it have?"        (safe, bound parameters)"
```

Notice there is **no** "write SQL" tool. The LLM picks a report and supplies *filter values* (e.g. `{"symbol": "AAPL"}`); `run_report` validates those fields against the report's schema and binds the values as parameters. The model never emits raw SQL, so a hostile value like `AAPL' OR '1'='1` is compared literally instead of being injected into the query. This mirrors the production tool in `src/tools/extraction_tools.py`.

In [ ]:
import json
import csv
import io
from typing import Dict, Optional
from strands import tool


@tool
def get_report_list() -> str:
    """Get the list of available market data reports with descriptions.

    Returns:
        str: JSON array of report objects with name and description.
    """
    return json.dumps(AVAILABLE_REPORTS, indent=2)


@tool
def get_report_schema(report_name: str) -> str:
    """Get the column schema for a report so you can choose filter fields.

    Args:
        report_name: Name of the report (e.g. 'TradeActivity', 'PriceBars').

    Returns:
        str: JSON object with parameter and column definitions.
    """
    schema = REPORT_SCHEMAS.get(report_name)
    if not schema:
        raise ValueError(f"Unknown report: {report_name}. Available: {list(REPORT_SCHEMAS.keys())}")
    return json.dumps(schema, indent=2)


@tool
def run_report(report_name: str, filters: Dict[str, str], limit: Optional[int] = None) -> dict:
    """Run a predefined report with schema-validated, parameter-bound filters.

    The LLM never writes SQL. It names a report and supplies equality filters;
    this tool validates every filter against the report's column allowlist and
    binds the values as named parameters. A hostile value such as
    "AAPL' OR '1'='1" is therefore compared literally, never injected.

    Args:
        report_name: A report from get_report_list (e.g. 'TradeActivity').
        filters: Equality filters keyed by column, e.g.
                 {"symbol": "AAPL", "date": "2024-03-15"}.
        limit: Optional row cap (1..10000).

    Returns:
        dict: {'success': bool, 'data': str (CSV), 'error': str or None, 'sql': str}
    """
    schema = REPORT_SCHEMAS.get(report_name)
    if not schema:
        return {"success": False, "data": "", "error": f"Unknown report: {report_name}"}

    # 1) Allowlist - reject any filter field that is not a column of this report.
    allowed = {c["name"] for c in schema["columns"]}
    unknown = set(filters) - allowed
    if unknown:
        return {"success": False, "data": "",
                "error": f"Unknown filter field(s) {sorted(unknown)}. Allowed: {sorted(allowed)}"}

    # 2) Build SQL with NAMED BIND PARAMETERS (:symbol, :date) - values are
    #    never string-interpolated into the query text.
    where = " AND ".join(f"{field} = :{field}" for field in filters)
    sql = f"SELECT * FROM {schema['reportName']}"
    if where:
        sql += f" WHERE {where}"
    if limit is not None:
        if not isinstance(limit, int) or not 1 <= limit <= 10_000:
            return {"success": False, "data": "", "error": "limit must be an integer in [1, 10000]"}
        sql += f" LIMIT {limit}"

    # 3) Execute against the mock data, treating the bound values as literals.
    #    In production you would hand (sql, filters) to a parameterised driver
    #    (SQLAlchemy text().bindparams, psycopg, or Athena ExecutionParameters).
    rows = list(csv.DictReader(io.StringIO(MOCK_DATA[report_name])))
    matched = [r for r in rows if all(str(r.get(k)) == str(v) for k, v in filters.items())]
    if limit is not None:
        matched = matched[:limit]

    out = io.StringIO()
    fieldnames = list(rows[0].keys()) if rows else []
    writer = csv.DictWriter(out, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(matched)
    return {"success": True, "data": out.getvalue(), "error": None, "sql": sql}


print("Tools defined: get_report_list, get_report_schema, run_report")

Tools defined: get_report_list, get_report_schema, run_report


## 4. Your First Surveillance Agent

Now we give these tools to a **Strands Agent** backed by Claude Sonnet 4.6 on Bedrock. The agent is defined by its system prompt and the tools it can access. It decides which tools to call and chains them together to answer the user's question.

We enable **thinking** so the agent reasons step-by-step before and between tool calls - this improves accuracy on analytical tasks.

In [ ]:
from strands import Agent
from strands.models.bedrock import BedrockModel
from botocore.config import Config

# MODEL_ID and region are defined once at the top (Section 1, Setup), where we
# also verify the model is reachable from your region. Part 1 uses Sonnet 4.6
# for fast, low-cost iteration; the deployed src/ system uses Opus 4.8.

# Specialist model WITH extended thinking. Sonnet 4.6 uses *adaptive* thinking:
# the model decides when and how much to think, and `effort` tunes the depth.
analyst_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=region,
    max_tokens=16000,
    boto_client_config=Config(
        retries={"max_attempts": 3, "mode": "adaptive"},
        read_timeout=300,
    ),
    additional_request_fields={
        "thinking": {"type": "adaptive"},
        "output_config": {"effort": "high"},
    },
)

# Faster model WITHOUT thinking - used for routing and synthesis, where we
# want speed and deterministic output over deep reasoning.
fast_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=region,
    temperature=0.0,
    max_tokens=4000,
)

print(f"Models configured (region={region}, model={MODEL_ID})")


Models configured (region=us-east-1, model=us.anthropic.claude-sonnet-4-6)


### Define the system prompt

The system prompt tells the agent what it is, what tools it has, and how to use them. The three-step tool pipeline is spelled out so the agent follows it consistently.

In [ ]:
ANALYST_PROMPT = '''You are a Security Monitor Agent that analyses single-day market data.

## Workflow
1. Call get_report_list to discover what data is available.
2. Pick the right report for the question and call get_report_schema to see its columns.
3. Call run_report with the report name and your filters
   (e.g. {"symbol": "AAPL", "date": "2024-03-15"}).
4. Analyse the rows it returns and present findings with specific numbers.

## Output
Return a JSON block with your findings:
```json
{
  "objective": "What the user asked",
  "business_insights": "Your findings backed by data"
}
```

Stay within the available data. If the data cannot answer the question, say so.'''

# Create the agent - Strands wires the tools, model, and prompt together
analyst = Agent(
    model=analyst_model,
    system_prompt=ANALYST_PROMPT,
    tools=[get_report_list, get_report_schema, run_report],
)

print("Analyst agent created")

Analyst agent created


### Run the agent

Ask a question and watch the agent reason through its tool pipeline. The default callback handler prints tool calls and text as they happen.

In [7]:
result = analyst("What trades happened on AAPL on March 15, 2024?")

The user wants to know about trades on AAPL on March 15, 2024. Let me start by discovering available reports and getting the schema for the relevant report simultaneously. Actually, I should first get the report list, then pick the right report and get its schema. But I can do both get_report_list and potentially guess the schema at the same time... Actually, let me first get the report list to see what's available.

Wait, I can call get_report_list first, and since I don't know what reports are available yet, I need to do that first before I can call get_report_schema. Let me start with get_report_list.Sure! Let me start by discovering what reports are available.
Tool #1: get_report_list
The **TradeActivity** report looks perfect for this question. Let me fetch its schema and the actual data at the same time!
Tool #2: get_report_schema

Tool #3: run_report
Let me analyze the data:

Trades on AAPL on March 15, 2024:
1. 09:35:00 - BUY 1500 @ $172.50 by ALPHA_CAPITAL
2. 09:35:02 - BUY 80

### Stream Agent Output

The cell above used the **default callback handler**, which prints as the agent works. For a real UI you would need the raw event stream instead. `stream_async` yields typed events - text chunks, tool calls, and reasoning tokens - as they happen.

We iterate those events by hand below. This is the exact pattern each LangGraph node will reuse to forward agent events into the graph's stream (Section 6).

In [8]:
# A fresh agent (agents accumulate messages across calls), with the default
# printing suppressed so we can handle the event stream ourselves.
analyst_streaming = Agent(
    model=analyst_model,
    system_prompt=ANALYST_PROMPT,
    tools=[get_report_list, get_report_schema, run_report],
    callback_handler=None,
)

previous_tool = None
async for event in analyst_streaming.stream_async("Show me the price bars for AAPL on March 15, 2024"):
    if "data" in event:
        print(event["data"], end="", flush=True)
    elif "reasoningText" in event:
        pass  # hide thinking tokens for cleaner output
    elif event.get("current_tool_use", {}).get("name"):
        tool_name = event["current_tool_use"]["name"]
        if tool_name != previous_tool:
            print(f"\n  >> tool: {tool_name}")
            previous_tool = tool_name

Sure! Let me start by discovering the available reports and fetching the schema for `PriceBars` at the same time!
  >> tool: get_report_list

  >> tool: get_report_schema
`PriceBars` is the right report and I have the schema. Now let me run it for **AAPL on 2024-03-15**!
  >> tool: run_report
Here are the results! Let me break down the findings:

---

### 📊 AAPL Price Bars — March 15, 2024

| Time | High | Low | Volume | # Trades | VWAP |
|----------|--------|--------|--------|----------|--------|
| 09:30 AM | 172.80 | 171.90 | 15,000 | 45 | 172.35 |
| 10:00 AM | 173.20 | 172.10 | 22,000 | 67 | 172.65 |
| 10:30 AM | 173.50 | 172.80 | 18,000 | 52 | 173.15 |
| 11:00 AM | 173.80 | 173.20 | 14,000 | 41 | 173.50 |

---

```json
{
  "objective": "Show price bars for AAPL on March 15, 2024",
  "business_insights": "AAPL exhibited a steady intraday uptrend on March 15, 2024. The day's low was $171.90 (at open) and the high reached $173.80 by 11:00 AM — a $1.90 range (+1.1%). The busiest bar wa

## 5. Add an Orchestrator

A single agent works for one kind of question, but real workloads are mixed. We add an **orchestrator** that reads each query and decides which specialist(s) should handle it.

We do this with an explicit routing node rather than the "agents as tools" pattern (where each specialist is wrapped as a tool the model can call). For surveillance work, explicit routing buys us three things:

- **Control & auditability** - routing is a visible decision in the graph, not an opaque tool call, so every run shows exactly which specialists were engaged.
- **Parallelism & ordering** - the graph can fan out to several specialists and gather their results, which is awkward when one agent calls another as a tool.
- **Isolation** - each specialist runs with its own prompt, tools, and context window, instead of competing for one agent's context.

Our orchestrator has no tools; it just emits a JSON routing decision that the graph acts on.

In [ ]:
ORCHESTRATOR_PROMPT = '''You are an orchestrator for a market surveillance system.
Analyse the user query and decide whether it needs data analysis.

If YES, respond with ONLY this JSON (no other text):
```json
{
  "agent_tasks": [
    {"agent": "analyst", "task": "specific analysis task description"}
  ]
}
```

If NO (e.g. a greeting or general knowledge question), respond conversationally.

RULES:
- One task per agent. Keep it focused.
- Include specific dates and securities mentioned in the query.
- If a required security code is missing, ask the user for it.'''

# callback_handler=None so the agent does not auto-print while streaming --
# we print the final routing decision once, below.
orchestrator = Agent(
    model=fast_model,
    system_prompt=ORCHESTRATOR_PROMPT,
    tools=[],  # no tools - the orchestrator only routes
    callback_handler=None,
)

# Test: this should produce a JSON routing decision
result = orchestrator("What was the trading volume of AAPL on March 15, 2024?")
print("--- Routing decision ---")
print(result)

--- Routing decision ---
```json
{
  "agent_tasks": [
    {"agent": "analyst", "task": "Retrieve and report the trading volume of AAPL (Apple Inc.) on March 15, 2024."}
  ]
}
```



## 6. Wire It Together with LangGraph

Calling agents by hand does not scale: as you add specialists you accumulate brittle `if/else` glue, there is no shared state, and nothing records *which* agents ran or lets you resume a conversation.

**LangGraph** replaces that glue with a **state graph**. Each agent becomes a node; edges describe the allowed transitions; a shared `State` dict flows through them; and a checkpointer persists that state so runs are resumable. The orchestrator decides the route, conditional edges pick the next node, and `get_stream_writer()` bridges each Strands agent's streaming events into LangGraph's custom stream.

```
START -> orchestrator -+- needs analysis -> analyst -> synthesizer -> END
                       +- no analysis ----------------------------- END
```

The key integration pattern lives inside each node: we iterate the Strands agent's `stream_async` events and forward them with `writer()`. Callers of `graph.astream(..., stream_mode="custom")` then receive real-time events from every agent.

### Add short-term memory with AgentCore Memory

Before we compile the graph, we create the **memory** it will checkpoint into. We use **Amazon Bedrock AgentCore Memory** for short-term state: `AgentCoreMemorySaver` (from `langgraph-checkpoint-aws`) is a LangGraph checkpointer that saves the graph state to AgentCore after every node. Conversations therefore survive restarts and can be resumed by `thread_id`.

Creating a memory resource takes a minute or two, so we do it once in its own cell and wait for it to become `ACTIVE`.

In [ ]:
# - Create an AgentCore Memory resource -------------------------
# Short-term memory: LangGraph checkpoints the graph state into it after every
# node, so a conversation can be resumed later by thread_id.
import time

control_client = boto3.client("bedrock-agentcore-control", region_name=region)

MEMORY_NAME = "WorkshopSurveillanceMemory"
MEMORY_ID = None

try:
    memory_response = control_client.create_memory(
        name=MEMORY_NAME,
        description="Short-term memory for the workshop LangGraph workflow.",
        eventExpiryDuration=7,  # days
    )
    MEMORY_ID = memory_response["memory"]["id"]
    print(f"Created AgentCore Memory: {MEMORY_ID}")
except control_client.exceptions.ValidationException:
    # A memory with this name already exists (the name is encoded in its id).
    # Reuse it instead of failing - this makes the cell safe to re-run.
    for mem in control_client.list_memories().get("memories", []):
        if MEMORY_NAME in (mem.get("id") or "") or MEMORY_NAME in (mem.get("name") or ""):
            MEMORY_ID = mem["id"]
            print(f"Using existing AgentCore Memory: {MEMORY_ID}")
            break

if MEMORY_ID is None:
    raise RuntimeError(f"Could not create or find memory {MEMORY_NAME!r}")

# Wait for ACTIVE, with a 10-minute deadline. Creation normally takes 1-3 min;
# the deadline stops the loop hanging on a stuck or FAILED resource.
deadline = time.time() + 600
while True:
    status = control_client.get_memory(memoryId=MEMORY_ID)["memory"]["status"]
    if status == "ACTIVE":
        break
    if status == "FAILED" or time.time() >= deadline:
        raise RuntimeError(f"Memory {MEMORY_ID} is {status!r} (expected ACTIVE)")
    time.sleep(10)
print("Memory is ACTIVE")

Using existing AgentCore Memory: WorkshopSurveillanceMemory-bFfdXy3OJz
Memory is ACTIVE


### Build the graph

Now we define the shared state, the three async nodes (orchestrator, analyst, synthesizer), the routing function, and the edges - then compile the graph using the memory resource above as its checkpointer. Each node returns **only the fields it sets**; LangGraph merges those partial updates into the shared state.

In [ ]:
from typing import TypedDict, Optional
from langgraph.graph import END, StateGraph
from langgraph.config import get_stream_writer
from langgraph_checkpoint_aws import AgentCoreMemorySaver


# - State schema ------------------------------------------------
# The shared state flows through every node. total=False lets each node return
# only the keys it sets (a partial update), which LangGraph merges in.
class SurveillanceState(TypedDict, total=False):
    query: str
    needs_analysis: Optional[bool]
    agent_task: Optional[str]
    analyst_result: Optional[str]
    final_response: Optional[str]


# - Node: Orchestrator ------------------------------------------
async def orchestrator_node(state: SurveillanceState) -> dict:
    writer = get_stream_writer()
    writer({"type": "agent_execution", "agent": "orchestrator", "status": "started"})

    orch = Agent(model=fast_model, system_prompt=ORCHESTRATOR_PROMPT, tools=[], callback_handler=None)
    chunks = []
    async for event in orch.stream_async(state["query"]):
        if "data" in event:
            writer({"type": "text", "content": event["data"], "agent": "orchestrator"})
            chunks.append(event["data"])

    response = "".join(chunks)
    writer({"type": "agent_execution", "agent": "orchestrator", "status": "completed"})

    # Parse the routing decision.
    needs_analysis, agent_task = False, None
    if "```json" in response:
        try:
            parsed = json.loads(response.split("```json")[1].split("```")[0])
            tasks = parsed.get("agent_tasks", [])
            if tasks:
                needs_analysis = True
                agent_task = tasks[0]["task"]
        except (json.JSONDecodeError, IndexError, KeyError):
            pass

    # Return ONLY the fields this node sets - LangGraph merges partial updates.
    return {"needs_analysis": needs_analysis, "agent_task": agent_task}


# - Node: Analyst -----------------------------------------------
async def analyst_node(state: SurveillanceState) -> dict:
    writer = get_stream_writer()
    writer({"type": "agent_execution", "agent": "analyst", "status": "started"})

    agent = Agent(
        model=analyst_model,
        system_prompt=ANALYST_PROMPT,
        tools=[get_report_list, get_report_schema, run_report],
        callback_handler=None,
    )

    prompt = state.get("agent_task") or state["query"]
    chunks, prev_tool = [], None
    async for event in agent.stream_async(prompt):
        if "data" in event:
            writer({"type": "text", "content": event["data"], "agent": "analyst"})
            chunks.append(event["data"])
        elif event.get("current_tool_use", {}).get("name"):
            tool_name = event["current_tool_use"]["name"]
            if tool_name != prev_tool:
                writer({"type": "tool", "name": tool_name, "agent": "analyst", "status": "started"})
                prev_tool = tool_name

    writer({"type": "agent_execution", "agent": "analyst", "status": "completed"})
    return {"analyst_result": "".join(chunks)}


# - Node: Synthesizer ------------------------------------------
async def synthesizer_node(state: SurveillanceState) -> dict:
    writer = get_stream_writer()
    writer({"type": "agent_execution", "agent": "synthesizer", "status": "started"})

    synth = Agent(
        model=fast_model,
        system_prompt="Synthesise analyst findings into a clear, structured summary with bullet points.",
        tools=[],
        callback_handler=None,
    )

    prompt = f"Original question: {state['query']}\n\nAnalyst findings:\n{state.get('analyst_result', '')}"
    chunks = []
    async for event in synth.stream_async(prompt):
        if "data" in event:
            writer({"type": "text", "content": event["data"], "agent": "synthesizer"})
            chunks.append(event["data"])

    writer({"type": "agent_execution", "agent": "synthesizer", "status": "completed"})
    return {"final_response": "".join(chunks)}


# - Routing function --------------------------------------------
def should_analyse(state: SurveillanceState) -> str:
    return "analyst" if state.get("needs_analysis") else END


# - Build the graph ---------------------------------------------
workflow = StateGraph(SurveillanceState)
workflow.add_node("orchestrator", orchestrator_node)
workflow.add_node("analyst", analyst_node)
workflow.add_node("synthesizer", synthesizer_node)

workflow.set_entry_point("orchestrator")
workflow.add_conditional_edges("orchestrator", should_analyse, {"analyst": "analyst", END: END})
workflow.add_edge("analyst", "synthesizer")
workflow.add_edge("synthesizer", END)

# Use the AgentCore Memory resource (created above) as the checkpointer.
checkpointer = AgentCoreMemorySaver(MEMORY_ID, region_name=region)
graph = workflow.compile(checkpointer=checkpointer)
print("LangGraph workflow compiled with AgentCore Memory checkpointer")

/Users/wangtenl/.pyenv/versions/3.13.7/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


LangGraph workflow compiled with AgentCore Memory checkpointer


### Run the Workflow

`graph.astream(..., stream_mode="custom")` delivers every `writer()` call from inside the nodes as a real-time chunk. This is exactly how the Streamlit UI and the AgentCore SSE endpoint stream output later - the same event vocabulary (`agent_execution`, `text`, `tool`) flows all the way through. We pass a `thread_id` so the checkpointer can persist and resume this conversation.

In [13]:
async for chunk in graph.astream(
    {"query": "What was the trading volume for AAPL on March 15, 2024?"},
    config={"configurable": {"thread_id": "workshop-001", "actor_id": "workshop-user"}},
    stream_mode="custom",
):
    ctype = chunk.get("type", "")
    agent = chunk.get("agent", "")

    if ctype == "agent_execution":
        icon = ">" if chunk["status"] == "started" else "ok"
        print(f"\n{icon} {agent}")
    elif ctype == "text":
        print(chunk["content"], end="", flush=True)
    elif ctype == "tool":
        print(f"\n  [tool] {chunk['name']}")


> orchestrator
```json
{
  "agent_tasks": [
    {"agent": "analyst", "task": "Retrieve and report the trading volume for AAPL (Apple Inc.) on March 15, 2024."}
  ]
}
```
ok orchestrator

> analyst
Sure! Let me start by discovering the available reports and their schemas. I'll begin with **Step 1** — fetching the report list.
  [tool] get_report_list
**Step 2** — The `PriceBars` report looks perfect for volume data. Let me fetch its schema to understand the available filters.
  [tool] get_report_schema
**Step 3** — Schema confirmed. Now let me run the report for `AAPL` on `2024-03-15`.
  [tool] run_report
**Step 4** — Data retrieved! Here's the full analysis:

---

### 📊 AAPL Trading Volume — March 15, 2024

| Time | High | Low | Volume | # Trades | VWAP |
|------|------|-----|--------|----------|------|
| 09:30 | $172.80 | $171.90 | 15,000 | 45 | $172.35 |
| 10:00 | $173.20 | $172.10 | 22,000 | 67 | $172.65 |
| 10:30 | $173.50 | $172.80 | 18,000 | 52 | $173.15 |
| 11:00 | $173.80 | $1

> **What just happened?** The orchestrator parsed the query, decided it needed analysis, and routed to the analyst. The analyst called three tools in sequence (`get_report_list` -> `get_report_schema` -> `run_report`), analysed the rows, and passed its findings to the synthesizer for a final summary. Every step streamed in real time through LangGraph.

You now have a working multi-agent workflow. In Part 2 we scale this to the full six-agent system and deploy it to AgentCore.

---
# Part 2 - Scale Up & Deploy to AgentCore

The mini-workflow you just built demonstrates the core patterns. The full system in `src/` extends it with:

| Feature | Part 1 (mini) | Full system (`src/`) |
|---------|--------------|---------------------|
| Specialist agents | 1 analyst | 4 (security, broker, risk, intel) |
| Routing | Fixed: analyst or nothing | Dynamic: any combination of agents |
| Data tables | 2 | 7 (trades, price bars, broker activity, risk scores, order book, daily snapshots, volume profile) |
| Session memory | AgentCore Memory (short-term) | MemorySaver + SessionManager |
| Agent state | Fresh each node | Restorable from previous messages |
| Deployment | Local only | AgentCore Runtime (managed containers) |

## 7. The Full Multi-Agent Workflow

Let's import the workflow and inspect its graph. It lives in `src/agents/workflow.py` and follows the same LangGraph pattern you just built, scaled to four specialists plus an orchestrator and synthesizer.

Each specialist is its own class (`SecurityMonitorAgent`, `BrokerMonitorAgent`, ...) that sets its model, system prompt, and tools. `Workflow` builds them through `AgentFactory.create_agent(name, ...)`, which looks the class up by name and - when a session has prior messages - restores them so the agent keeps its context across turns.

The orchestrator routes each query to any combination of specialists; they can hand off to one another and finally converge on the synthesizer, which writes the report. The compiled graph looks like this:

![Multi-agent orchestration graph](../langgraph-architecture.svg)


In [26]:
import sys, os
from pathlib import Path


from src.agents import Workflow
from IPython.display import Image, display

full_workflow = Workflow()

# To see LangGraph's own diagram, uncomment the line below:
#display(Image(full_workflow.graph.get_graph().draw_mermaid_png()))


## 8. The AgentCore Entry Point

To run on AgentCore we wrap the workflow in a `BedrockAgentCoreApp`. The `@app.entrypoint` decorator marks the async function AgentCore invokes for each request: it receives the JSON payload and yields streaming chunks. The real entry point is [`api.py`](../api.py) in the project root - open it to see the production version (logging, error handling, `session_id` propagation). It is intentionally tiny:

```python
# api.py (simplified)
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from src.agents import Workflow

app = BedrockAgentCoreApp()
workflow = Workflow()

@app.entrypoint
async def market_surveillance_workflow(payload):
    prompt = payload.get("prompt")
    session_id = payload.get("session_id", "default-session")
    async for chunk in workflow.stream_query(session_id=session_id, prompt=prompt):
        yield chunk

if __name__ == "__main__":
    app.run()
```

**What happens behind the scenes?** `BedrockAgentCoreApp` runs a small HTTP server on port 8080, exposes the `/invocations` endpoint AgentCore calls for each request, and adds a `/ping` health check - so you write an async function, not a web service. The `Dockerfile` packages this with all dependencies into an ARM64 container image; AgentCore handles orchestration, scaling, and networking.

## 9. Deploy to AgentCore Runtime

We use the `bedrock-agentcore-starter-toolkit` to deploy in one step. It:

1. Uploads your source to S3
2. Triggers an AWS CodeBuild job that compiles an ARM64 container image
3. Pushes the image to ECR
4. Creates (or updates) an AgentCore Runtime endpoint

> **Expected time: 5-10 minutes** for the first build. Subsequent runs reuse the existing runtime.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
import yaml

AGENT_NAME = "market_surveillance_workflow"

agentcore_runtime = Runtime()

agentcore_runtime.configure(
    entrypoint="api.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=AGENT_NAME,
)

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "api.py").exists() else Path.cwd().parent

# Guard against a stale local config. .bedrock_agentcore.yaml may pin an
# agent_id from a previous run that was since deleted (e.g. by the Cleanup
# section). With auto_update_on_conflict=True the toolkit would then call
# UpdateAgentRuntime on a runtime that no longer exists and fail with
# ResourceNotFoundException. So we verify the recorded runtime still exists
# and, if not, clear the stale IDs so launch() creates a fresh one.
_cfg_path = PROJECT_ROOT / ".bedrock_agentcore.yaml"
_cfg = yaml.safe_load(_cfg_path.read_text())
_bac = _cfg.get("agents", {}).get(AGENT_NAME, {}).get("bedrock_agentcore", {})
_stale_id = _bac.get("agent_id")
if _stale_id:
    _control = boto3.client("bedrock-agentcore-control", region_name=region)
    try:
        _control.get_agent_runtime(agentRuntimeId=_stale_id)
        print(f"Found existing runtime {_stale_id} - will update it.")
    except _control.exceptions.ResourceNotFoundException:
        print(f"Recorded runtime {_stale_id} no longer exists - creating a fresh one.")
        _bac["agent_id"] = None
        _bac["agent_arn"] = None
        _cfg_path.write_text(yaml.safe_dump(_cfg, sort_keys=False))

launch_result = agentcore_runtime.launch(auto_update_on_conflict=True)

print(f"Agent ARN : {launch_result.agent_arn}")
print(f"Agent ID  : {launch_result.agent_id}")
print(f"ECR URI   : {launch_result.ecr_uri}")


Entrypoint parsed: file=/Users/ggeinke/PycharmProjects/lg-strands-blog/api.py, bedrock_agentcore_name=api
Memory configured with STM only
Configuring BedrockAgentCore agent: market_surveillance_workflow
Will create new memory with mode: STM_ONLY
Memory configuration: Short-term memory only
Found existing memory ID from previous launch: market_surveillance_workflow_mem-rXr6iT9sZi
Generated Dockerfile: Dockerfile
Generated .dockerignore: /Users/ggeinke/PycharmProjects/lg-strands-blog/.dockerignore
Keeping 'market_surveillance_workflow' as default agent
Bedrock AgentCore configured: /Users/ggeinke/PycharmProjects/lg-strands-blog/.bedrock_agentcore.yaml
🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_bu

Recorded runtime market_surveillance_workflow-OwJMRa8Y5T no longer exists - creating a fresh one.


✅ MemoryManager initialized for region: us-west-2
🔎 Retrieving memory resource with ID: market_surveillance_workflow_mem-rXr6iT9sZi...
  Found memory: market_surveillance_workflow_mem-rXr6iT9sZi
Found existing memory in cloud: market_surveillance_workflow_mem-rXr6iT9sZi
Existing memory has 0 strategies
✅ Using existing STM-only memory
Starting CodeBuild ARM64 deployment for agent 'market_surveillance_workflow' to account 864981718505 (us-west-2)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: market_surveillance_workflow


Repository doesn't exist, creating new ECR repository: bedrock-agentcore-market_surveillance_workflow


✅ ECR repository available: 864981718505.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-market_surveillance_workflow
Getting or creating execution role for agent: market_surveillance_workflow
Using AWS region: us-west-2, account ID: 864981718505
Role name: AmazonBedrockAgentCoreSDKRuntime-us-west-2-68d70a95a7
✅ Reusing existing execution role: arn:aws:iam::864981718505:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-68d70a95a7
✅ Execution role available: arn:aws:iam::864981718505:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-68d70a95a7
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: market_surveillance_workflow
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-68d70a95a7
Reusing existing CodeBuild execution role: arn:aws:iam::864981718505:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-68d70a95a7
Using dockerignore.template with 45 patterns for zip filtering


## 10. Invoke the Deployed Agent

AgentCore returns a **server-sent events (SSE) stream**. The helper below parses each chunk and prints it as it arrives, tagging agent start/stop and tool calls. (The Streamlit client in `app.py` consumes the same stream to drive its UI.)

In [16]:
import json


def _parse_sse_line(line):
    """Parse one SSE line into a chunk dict, handling AgentCore's double-wrapped
    `data: "data: {...}"` framing. Returns None for keepalives / blank lines."""
    line = line.strip()
    if not line.startswith("data: "):
        return None
    inner = line[6:]
    try:
        unwrapped = json.loads(inner)  # strips the outer quoting
        if isinstance(unwrapped, str):
            unwrapped = unwrapped.strip()
            if unwrapped.startswith("data: "):
                unwrapped = unwrapped[6:].strip()
            return json.loads(unwrapped)
        if isinstance(unwrapped, dict):
            return unwrapped
    except (json.JSONDecodeError, TypeError):
        pass
    return None


def stream_response(boto3_response):
    """Stream an AgentCore SSE response and print each chunk as it arrives."""
    for raw in boto3_response["response"].iter_lines():
        line = raw.decode("utf-8") if isinstance(raw, bytes) else raw
        chunk = _parse_sse_line(line)
        if not isinstance(chunk, dict):
            continue
        ctype, agent = chunk.get("type", "text"), chunk.get("agent", "")
        if ctype == "agent_execution":
            mark = ">>" if chunk.get("status") == "started" else "ok"
            print(f"\n[{mark}] {agent}")
        elif ctype == "tool":
            print(f"\n   (tool) {chunk.get('name', '')}")
        elif ctype in ("text", "reasoningText"):
            content = chunk.get("content", "")
            if isinstance(content, list):
                content = "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in content)
            print(content, end="", flush=True)


print("Helper loaded: stream_response()")

Helper loaded: stream_response()


In [17]:
from botocore.config import Config

agentcore_client = boto3.client(
    "bedrock-agentcore",
    region_name=region,
    config=Config(read_timeout=600)  # 10 minutes
)


### Query 1 - Single-security intraday analysis

In [18]:
agent_arn = launch_result.agent_arn

stream_response(agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({
        "prompt": "What caused the price for AAPL to increase at 11:00 AM on March 15, 2024?",
        "session_id": "workshop-session-001",
    }),
))


[>>] orchestrator
```json
{
  "agent_tasks": [
    {"agent": "security_monitor", "task": "Analyze the price increase for AAPL at 11:00 AM on March 15, 2024, including intraday price movements and volume patterns around that time"},
    {"agent": "intel_analyst", "task": "Research external factors such as market news, regulatory filings, or public disclosures for AAPL around March 15, 2024 that may explain a price increase at 11:00 AM"}
  ]
}
```
[ok] orchestrator

[>>] security_monitor
I'll analyze AAPL's price movement around 11:00 AM on March 15, 2024. Let me start by reviewing available reports.
   (tool) get_report_list

   (tool) get_report_list

   (tool) get_report_list

   (tool) tool_result
This request is in scope — single security, single day, intraday analysis. Let me load the schemas for the most relevant reports.
   (tool) get_report_schema

   (tool) get_report_schema

   (tool) get_report_schema

   (tool) get_report_schema

   (tool) get_report_schema

   (tool) get_r

### Query 2 - Cross-security broker summary

In [19]:
stream_response(agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({
        "prompt": "Summarise broker activity across all securities on March 15, 2024.",
        "session_id": "workshop-session-002",
    }),
))


[>>] orchestrator
```json
{
  "agent_tasks": [
    {"agent": "risk_monitor", "task": "Summarise broker activity across all securities on March 15, 2024"}
  ]
}
```
[ok] orchestrator

[>>] risk_monitor
I'll analyze broker activity for March 15, 2024. Let me start by reviewing the available reports.
   (tool) get_report_list

   (tool) get_report_list

   (tool) get_report_list

   (tool) tool_result
The `BrokerActivity` report is the best fit for summarizing broker activity across all securities. Let me get its schema.
   (tool) get_report_schema

   (tool) get_report_schema

   (tool) get_report_schema

   (tool) tool_result
Now let me run the report for March 15, 2024 across all securities.
   (tool) run_report

   (tool) run_report

   (tool) run_report

   (tool) tool_result
Here is the summary of broker activity across all securities on March 15, 2024.

```json
{
  "objective": "Summarize aggregated broker trading activity (buy/sell volumes, trade counts, market share) across all 

### Query 3 - Conversational follow-up

Using the same `session_id` enables conversational memory. The orchestrator sees the previous query context and can answer follow-up questions without re-running the full analysis.

In [20]:
stream_response(agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({
        "prompt": "Which broker had the highest risk score?",
        "session_id": "workshop-session-002",  # same session as Query 2
    }),
))


[>>] orchestrator
I need a bit more information to route this correctly.

The risk score analysis can be handled by **broker_monitor** (single security across multiple days), but your query is missing a key detail: **which security** are you asking about?

Could you please provide the security code you'd like me to analyze? For example:
- "Which broker had the highest risk score for **AAPL** over the past month?"

Once you share the security, I'll route this to the appropriate specialist.
[ok] orchestrator


---
## 11. Cleanup

> **Run this section when you are finished** to avoid ongoing AWS charges.

The cells below delete the AgentCore Runtime and ECR repository. These are the resources that incur costs.

**Not deleted automatically** (use the AWS console if you no longer need them):
- IAM execution role
- CodeBuild project
- S3 source bucket
- Local `.bedrock_agentcore.yaml` file

In [21]:
import yaml

# Read IDs from toolkit config
with open(PROJECT_ROOT / ".bedrock_agentcore.yaml") as f:
    toolkit_config = yaml.safe_load(f)

agent_config = toolkit_config["agents"][AGENT_NAME]
agent_arn = agent_config["bedrock_agentcore"]["agent_arn"]
agent_id = agent_config["bedrock_agentcore"]["agent_id"]
ecr_repo_name = agent_config["aws"]["ecr_repository"].split("/")[-1]

print("About to delete:")
print(f"  AgentCore Runtime : {agent_id}")
print(f"  ECR repository    : {ecr_repo_name}")
print(f"  AgentCore Memory    : {MEMORY_ID}")

About to delete:
  AgentCore Runtime : market_surveillance_workflow-t7mUoSCD10
  ECR repository    : bedrock-agentcore-market_surveillance_workflow
  AgentCore Memory    : WorkshopSurveillanceMemory-bFfdXy3OJz


In [22]:
# Delete AgentCore Runtime
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=region)
agentcore_control.delete_agent_runtime(agentRuntimeId=agent_id)
print(f"Deleted AgentCore Runtime: {agent_id}")

# Delete ECR repository (force=True removes all images)
ecr_client = boto3.client("ecr", region_name=region)
ecr_client.delete_repository(repositoryName=ecr_repo_name, force=True)
print(f"Deleted ECR repository: {ecr_repo_name}")


# Delete AgentCore Memory (created in Part 1)
try:
    agentcore_control.delete_memory(memoryId=MEMORY_ID)
    print(f"Deleted AgentCore Memory: {MEMORY_ID}")
except Exception:
    pass  # memory may not exist if Part 1 was skipped

print("\nCleanup complete.")

Deleted AgentCore Runtime: market_surveillance_workflow-t7mUoSCD10
Deleted ECR repository: bedrock-agentcore-market_surveillance_workflow
Deleted AgentCore Memory: WorkshopSurveillanceMemory-bFfdXy3OJz

Cleanup complete.


---
## What You Built

| Step | What happened |
|------|--------------|
| **Part 1** - Tools | Defined `@tool` functions wrapping mock market data |
| **Part 1** - Agent | Created a Strands agent that chains tools to answer questions |
| **Part 1** - Streaming | Used `stream_async` to observe agent events in real time |
| **Part 1** - Orchestrator | Added a routing agent that parses queries into tasks |
| **Part 1** - LangGraph | Wired orchestrator -> analyst -> synthesizer into a state graph with `get_stream_writer()` bridging |
| **Part 2** - Full system | Imported the six-agent production workflow |
| **Part 2** - Deploy | One-command deployment to AgentCore Runtime via starter toolkit |
| **Part 2** - Invoke | Streamed multi-agent responses over SSE from a managed endpoint |

## Next Steps

- **Add a custom tool** - create a new `@tool` in `src/tools/` and register it with the relevant specialist agent
- **Connect real data** - replace `MOCK_DATA` with actual database queries
- **Enable authentication** - pass `authorizer_configuration` to `configure()` for JWT or Cognito-based access
- **Add a Streamlit UI** - run `streamlit run app.py` to use the included chat interface
- **Monitor with CloudWatch** - AgentCore emits metrics and logs; create a dashboard to track latency and errors